# Results Recommendation Audit

Purpose: review **every recommendation context** (not just settled BET rows) and surface weak pockets to attack.

Use this notebook when you want:
- a full recommendation ledger view,
- weak-point slices (side, rest, projected workload),
- actionable diagnostics before changing gates/calibration.

In [1]:
from pathlib import Path
import sys
import polars as pl
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "production").exists():
    for parent in ROOT.parents:
        if (parent / "production").exists() and (parent / "src").exists():
            ROOT = parent
            break

sys.path.insert(0, str(ROOT / "src"))
from Python.notebook_analysis_utils import (
    has_over_clv_red_flag,
    keep_best_available_lines,
)

LEDGER_PATH = ROOT / "artifacts" / "odds_log" / "ledger.parquet"
PROJ_PATH = ROOT / "artifacts" / "projection_log" / "projections.parquet"


def show_table(df: pl.DataFrame, n: int = 200):
    display(df.head(n).to_pandas().round(3))


if not LEDGER_PATH.exists():
    raise FileNotFoundError(f"Missing {LEDGER_PATH}")

ledger = pl.read_parquet(LEDGER_PATH)
print(f"ledger rows={ledger.height} date_range={ledger['game_date'].min()} -> {ledger['game_date'].max()}")

if PROJ_PATH.exists():
    proj = (
        pl.read_parquet(PROJ_PATH)
        .with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10))
        .select([c for c in [
            "game_date", "player_name", "days_rest", "projected_tbf", "expected_K",
            "is_out_of_support", "is_abbreviated_outing", "oos_reason"
        ] if c in pl.read_parquet(PROJ_PATH, n_rows=1).columns])
        .unique(subset=["game_date", "player_name"], keep="first")
    )
else:
    proj = pl.DataFrame()

print(f"projection rows={proj.height}")

ledger rows=802 date_range=2026-07-30 -> 2026-08-17
projection rows=511


In [2]:
base = ledger.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10))

if proj.height:
    base = base.join(proj, on=["game_date", "player_name"], how="left")

print("recommendation context mix")
summary = (
    base.group_by([c for c in ["snapshot", "status", "side", "passes_floor"] if c in base.columns])
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
)
show_table(summary, n=200)

cols = [
    c for c in [
        "game_date", "player_name", "book", "line", "side", "edge", "passes_floor", "status",
        "days_rest", "projected_tbf", "expected_K", "is_out_of_support", "oos_reason", "note"
    ] if c in base.columns
]
print("\nlatest recommendation rows (most recent 120)")
recent = base.sort([c for c in ["logged_at_utc", "game_date"] if c in base.columns], descending=True)
show_table(recent.select(cols), n=120)

recommendation context mix


,snapshot,status,side,passes_floor,n
0,bet,settled,under,False,265
1,bet,settled,over,False,229
2,bet,settled,under,True,166
3,bet,settled,over,True,132
4,bet,void,under,True,4
5,bet,void,over,True,4
6,bet,void,over,False,2



latest recommendation rows (most recent 120)


,game_date,player_name,book,line,side,edge,passes_floor,status,days_rest,projected_tbf,expected_K,note
0,2026-08-17,Nolan McLean,draftkings,5.5,under,0.092,False,settled,6.0,23.999,5.455,close_cross_book=1
1,2026-08-17,Blake Snell,draftkings,6.5,over,0.028,False,settled,6.0,23.463,7.208,close_unavailable=past_window_tip-15m
2,2026-08-17,Tomoyuki Sugano,draftkings,2.5,over,0.220,True,settled,6.0,23.243,4.233,close_unavailable=past_window_tip-15m
3,2026-08-17,Shota Imanaga,draftkings,5.5,over,0.071,False,settled,6.0,24.159,6.056,
4,2026-08-17,Luis Castillo,draftkings,4.5,under,0.086,False,settled,5.0,23.321,4.490,
...,...,...,...,...,...,...,...,...,...,...,...,...
115,2026-08-14,Kyle Freeland,draftkings,5.5,under,0.150,True,settled,6.0,23.764,4.382,
116,2026-08-14,Yoshinobu Yamamoto,fanduel,6.5,under,0.076,False,settled,6.0,25.245,6.466,
117,2026-08-14,Robert Gasser,fanduel,4.5,under,0.055,False,settled,6.0,22.334,4.266,
118,2026-08-14,Yoshinobu Yamamoto,draftkings,6.5,under,0.083,False,settled,6.0,25.245,6.466,


In [3]:
settled_raw = base.filter((pl.col("status") == "settled") & (pl.col("stake").fill_null(0) > 0))
settled = keep_best_available_lines(settled_raw)
print(f"settled_raw_n={settled_raw.height} settled_best_line_n={settled.height}")
if settled.is_empty():
    print("No settled staked rows yet.")
else:
    settled = settled.with_columns([
        pl.when(pl.col("days_rest").is_null()).then(pl.lit("unknown"))
        .when(pl.col("days_rest") < 10).then(pl.lit("rest_<10"))
        .when(pl.col("days_rest") < 45).then(pl.lit("rest_10_44"))
        .otherwise(pl.lit("rest_45_plus")).alias("rest_bucket"),
        pl.when(pl.col("projected_tbf").is_null()).then(pl.lit("unknown"))
        .when(pl.col("projected_tbf") < 12).then(pl.lit("tbf_<12"))
        .when(pl.col("projected_tbf") < 15).then(pl.lit("tbf_12_15"))
        .otherwise(pl.lit("tbf_15_plus")).alias("tbf_bucket"),
    ])

    def slice_report(by: str) -> pl.DataFrame:
        return (
            settled.group_by(by)
            .agg(
                pl.len().alias("n"),
                pl.col("stake").sum().alias("stake"),
                pl.col("pnl").sum().alias("pnl"),
                (pl.col("pnl").sum() / pl.col("stake").sum()).alias("roi"),
                pl.col("clv_pp").drop_nulls().mean().alias("mean_clv_pp"),
            )
            .sort("n", descending=True)
        )

    print("weak-point scan by side")
    side_health = slice_report("side")
    show_table(side_health, n=20)
    if has_over_clv_red_flag(side_health):
        print("RED FLAG: over mean_clv_pp <= 0 (or missing).")

    print("\nweak-point scan by rest bucket")
    show_table(slice_report("rest_bucket"), n=20)

    print("\nweak-point scan by projected_tbf bucket")
    show_table(slice_report("tbf_bucket"), n=20)

settled_raw_n=298 settled_best_line_n=164
weak-point scan by side


,side,n,stake,pnl,roi,mean_clv_pp
0,under,88,7030.658,952.251,0.135,0.005
1,over,76,5062.103,-879.670,-0.174,0.010



weak-point scan by rest bucket


,rest_bucket,n,stake,pnl,roi,mean_clv_pp
0,rest_<10,152,10915.920,522.978,0.048,0.011
1,rest_10_44,12,1176.841,-450.398,-0.383,-0.027



weak-point scan by projected_tbf bucket


,tbf_bucket,n,stake,pnl,roi,mean_clv_pp
0,tbf_15_plus,163,11917.978,-45.516,-0.004,0.009
1,tbf_12_15,1,174.783,118.097,0.676,-0.142


In [ ]:
# Slippage decomposition: open→bet vs bet→close CLV components
from Python.market import american_to_implied_prob, devig_two_way

def slippage_components(df: pl.DataFrame) -> pl.DataFrame:
    req = {"side", "bet_price", "other_price", "close_over", "close_under"}
    if not req.issubset(set(df.columns)):
        return pl.DataFrame()
    scoped = df.filter(
        pl.col("side").is_in(["over", "under"])
        & pl.col("bet_price").is_not_null()
        & pl.col("other_price").is_not_null()
        & pl.col("close_over").is_not_null()
        & pl.col("close_under").is_not_null()
    )
    rows = []
    for r in scoped.select("side", "bet_price", "other_price", "close_over", "close_under", "book", "line").to_dicts():
        side = str(r.get("side"))
        bp = float(r["bet_price"])
        op = float(r["other_price"])
        co = float(r["close_over"])
        cu = float(r["close_under"])
        try:
            p_open_over, p_open_under = devig_two_way(bp, op)
            p_close_over, p_close_under = devig_two_way(co, cu)
        except Exception:
            continue
        p_open = p_open_over if side == "over" else p_open_under
        p_close = p_close_over if side == "over" else p_close_under
        p_bet_single = american_to_implied_prob(bp)
        rows.append({
            "side": side,
            "book": r.get("book"),
            "line": r.get("line"),
            "open_to_bet_pp": p_open - p_bet_single,
            "bet_to_close_pp": p_close - p_open,
            "open_to_close_pp": p_close - p_bet_single,
        })
    return pl.DataFrame(rows)

slip = slippage_components(settled)
if slip.is_empty():
    print("No slippage decomposition rows available.")
else:
    overall = slip.select(
        pl.len().alias("n"),
        pl.col("open_to_bet_pp").mean().alias("mean_open_to_bet_pp"),
        pl.col("bet_to_close_pp").mean().alias("mean_bet_to_close_pp"),
        pl.col("open_to_close_pp").mean().alias("mean_open_to_close_pp"),
    )
    print("Overall slippage decomposition")
    show_table(overall)

    by_side = slip.group_by("side").agg(
        pl.len().alias("n"),
        pl.col("open_to_bet_pp").mean().alias("mean_open_to_bet_pp"),
        pl.col("bet_to_close_pp").mean().alias("mean_bet_to_close_pp"),
        pl.col("open_to_close_pp").mean().alias("mean_open_to_close_pp"),
    ).sort("side")
    print("\nBy side")
    show_table(by_side)

    by_book = slip.group_by("book").agg(
        pl.len().alias("n"),
        pl.col("open_to_bet_pp").mean().alias("mean_open_to_bet_pp"),
        pl.col("bet_to_close_pp").mean().alias("mean_bet_to_close_pp"),
        pl.col("open_to_close_pp").mean().alias("mean_open_to_close_pp"),
    ).sort("n", descending=True)
    print("\nBy book")
    show_table(by_book, n=20)

In [ ]:
# Regime-aware slippage: recent windows only
if slip.is_empty():
    print("No slippage rows.")
else:
    # Rebuild with required columns for recent-window summaries.
    required_cols = [
        "game_date",
        "side",
        "bet_price",
        "other_price",
        "close_over",
        "close_under",
        "book",
        "line",
    ]
    scoped = settled.select([c for c in required_cols if c in settled.columns])
    scoped = scoped.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("gdate"))

    def summarize_recent(n_rows: int | None):
        s = scoped if n_rows is None else scoped.tail(n_rows)
        tmp = slippage_components(s)
        if tmp.is_empty():
            return None
        return {
            "window": "full" if n_rows is None else f"last_{n_rows}",
            "n": int(tmp.height),
            "mean_open_to_bet_pp": float(tmp["open_to_bet_pp"].mean()),
            "mean_bet_to_close_pp": float(tmp["bet_to_close_pp"].mean()),
            "mean_open_to_close_pp": float(tmp["open_to_close_pp"].mean()),
        }

    rows = [r for r in [summarize_recent(None), summarize_recent(60), summarize_recent(30)] if r is not None]
    if rows:
        print("Regime-aware slippage summary (prefer recent windows for current process decisions):")
        show_table(pl.DataFrame(rows), n=10)